In [ ]:
!pip install -q "timm>=1.0.9"

## 1. Configuración

In [ ]:
import os, json, time, math, random, platform, zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler, Subset
from torchvision import datasets, transforms
from torchvision.transforms import v2

import timm
from timm.data import Mixup
from timm.loss import SoftTargetCrossEntropy
from timm.utils import ModelEmaV3
from sklearn.metrics import f1_score


class CFG:
    MODEL_VARIANT = "convnext_small_cbam"
    # convnext_small_cbam | convnext_small_plain | resnet50 | efficientnet_b3 | swin_tiny | mobilenetv3_large


    DATA_DIR   = "/kaggle/input/datasets/ibrahimagabardiop/niayes-crops-disease-v2"
    FROZEN_DIR = "/kaggle/input/datasets/emersonquispe/plantnova-particion-congelada"
    OUT_DIR    = "/kaggle/working"

    IMG_SIZE, RESIZE = 224, 256
    MEAN = (0.485, 0.456, 0.406)
    STD  = (0.229, 0.224, 0.225)

    BATCH_SIZE    = 64
    NUM_WORKERS   = os.cpu_count() or 4
    EPOCHS        = 30
    WARMUP_EPOCHS = 1
    LR_BACKBONE   = 1e-4
    LR_HEAD       = 1e-3
    MIN_LR_RATIO  = 0.01
    WEIGHT_DECAY  = 0.05
    GRAD_CLIP     = 1.0
    LABEL_SMOOTHING = 0.1
    DROPOUT    = 0.3
    DROP_PATH  = 0.2
    PATIENCE   = 8

    MIXUP_ALPHA      = 0.2
    CUTMIX_ALPHA     = 1.0
    MIXUP_PROB       = 0.8
    MIXUP_OFF_EPOCHS = 5

    BALANCED_SAMPLING = True
    SAMPLER_POWER     = 0.5     # peso = 1 / n^power; 0.5 = raiz cuadrada inversa
    EMA_DECAY = 0.9998

    EVAL_TRAIN_LIMPIO = True    # evalua train sin aumento ni MixUp
    RESUME_PATH = None
    SEED = 42


def seed_everything(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

seed_everything(CFG.SEED)

# Reproducibilidad del orden de los datos: generador propio y semilla por worker
g_loader = torch.Generator(); g_loader.manual_seed(CFG.SEED)

def seed_worker(worker_id):
    s = torch.initial_seed() % 2**32
    np.random.seed(s); random.seed(s)

os.makedirs(CFG.OUT_DIR, exist_ok=True)

device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"
torch.backends.cudnn.benchmark = True

gpu_name = torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU"
gpu_mem  = (torch.cuda.get_device_properties(0).total_memory / 1e9
            if device.type == "cuda" else 0)
print(f"Variante: {CFG.MODEL_VARIANT}")
print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB) | torch {torch.__version__} | timm {timm.__version__}")

## 2. Registro de variantes


In [ ]:
VARIANTES = {
    "convnext_small_cbam":  {"timm": "convnext_small.in12k_ft_in1k",          "cbam": True},
    "convnext_small_plain": {"timm": "convnext_small.in12k_ft_in1k",          "cbam": False},
    "resnet50":             {"timm": "resnet50.a1_in1k",                      "cbam": False},
    "efficientnet_b3":      {"timm": "efficientnet_b3.ra2_in1k",              "cbam": False},
    "swin_tiny":            {"timm": "swin_tiny_patch4_window7_224.ms_in1k",  "cbam": False},
    "mobilenetv3_large":    {"timm": "mobilenetv3_large_100.ra_in1k",         "cbam": False},
}
assert CFG.MODEL_VARIANT in VARIANTES, f"Variante desconocida: {CFG.MODEL_VARIANT}"
VARIANTE = VARIANTES[CFG.MODEL_VARIANT]


class CBAM(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(nn.Conv2d(channels, channels // reduction, 1, bias=False),
                                nn.ReLU(),
                                nn.Conv2d(channels // reduction, channels, 1, bias=False))
        self.sigmoid  = nn.Sigmoid()
        self.conv_spa = nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False)

    def forward(self, x):
        x = x * self.sigmoid(self.fc(self.avg_pool(x)) + self.fc(self.max_pool(x)))
        avg_s = torch.mean(x, dim=1, keepdim=True)
        max_s, _ = torch.max(x, dim=1, keepdim=True)
        return x * self.sigmoid(self.conv_spa(torch.cat([avg_s, max_s], dim=1)))


class Clasificador(nn.Module):
    def __init__(self, timm_name, num_classes, usa_cbam, pretrained=True,
                 drop_path=0.0, dropout=0.3):
        super().__init__()
        self.usa_cbam = usa_cbam
        if usa_cbam:
            self.backbone = timm.create_model(timm_name, pretrained=pretrained, num_classes=0,
                                              global_pool="", drop_path_rate=drop_path)
        else:
            self.backbone = timm.create_model(timm_name, pretrained=pretrained, num_classes=0,
                                              drop_path_rate=drop_path)

        with torch.no_grad():                      
            s = self.backbone(torch.zeros(1, 3, CFG.IMG_SIZE, CFG.IMG_SIZE))
        if usa_cbam:
            assert s.ndim == 4 and s.shape[2] == s.shape[3], f"Mapa inesperado: {tuple(s.shape)}"
            self.num_features = s.shape[1]
            self.cbam = CBAM(self.num_features, reduction=16)
            self.pool = nn.AdaptiveAvgPool2d(1)
        else:
            assert s.ndim == 2, f"Se esperaba un vector agrupado: {tuple(s.shape)}"
            self.num_features = s.shape[1]         

        self.classifier = nn.Sequential(nn.LayerNorm(self.num_features),
                                        nn.Dropout(p=dropout),
                                        nn.Linear(self.num_features, num_classes))

    def forward(self, x):
        x = self.backbone(x)
        if self.usa_cbam:
            x = self.pool(self.cbam(x)).flatten(1)
        return self.classifier(x)

## 3. Transformaciones

In [ ]:
_fill = tuple(int(m * 255) for m in CFG.MEAN)

train_ops = [
    v2.ToImage(),
    v2.RandomZoomOut(fill=_fill, side_range=(1.0, 1.8), p=0.2),
    v2.RandomResizedCrop(CFG.IMG_SIZE, scale=(0.3, 1.0), antialias=True),
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomVerticalFlip(p=0.2),
    v2.RandomApply([v2.RandomRotation(30)], p=0.5),
    v2.RandomPerspective(distortion_scale=0.2, p=0.2),
    v2.RandomApply([v2.ColorJitter(brightness=0.4, contrast=0.4,
                                   saturation=0.3, hue=0.05)], p=0.8),
    v2.RandomGrayscale(p=0.05),
    v2.RandomApply([v2.GaussianBlur(kernel_size=9, sigma=(0.1, 3.0))], p=0.3),
]
TIENE_JPEG = hasattr(v2, "JPEG")
if TIENE_JPEG:
    train_ops.append(v2.RandomApply([v2.JPEG(quality=(30, 95))], p=0.3))
else:
    print("torchvision sin v2.JPEG: se omite la compresión JPEG")
train_ops += [
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(CFG.MEAN, CFG.STD),
    v2.RandomErasing(p=0.25, scale=(0.02, 0.2), ratio=(0.3, 3.3), value=0),
]
train_tf = v2.Compose(train_ops)

val_tf = transforms.Compose([
    transforms.Resize(CFG.RESIZE), transforms.CenterCrop(CFG.IMG_SIZE),
    transforms.ToTensor(), transforms.Normalize(CFG.MEAN, CFG.STD),
])

## 4. Partición congelada


In [ ]:
train_full = datasets.ImageFolder(os.path.join(CFG.DATA_DIR, "train"), transform=train_tf)
val_ds     = datasets.ImageFolder(os.path.join(CFG.DATA_DIR, "val"),   transform=val_tf)
test_ds    = datasets.ImageFolder(os.path.join(CFG.DATA_DIR, "test"),  transform=val_tf)

class_names = train_full.classes
num_classes = len(class_names)
assert val_ds.classes == class_names == test_ds.classes, "Las clases difieren entre particiones"

keep_path = os.path.join(CFG.FROZEN_DIR, "keep_indices_train.npy")
if not os.path.exists(keep_path):
    raise FileNotFoundError(
        f"No se encontró {keep_path}.\n"
        "Ejecuta primero la Fase 0 y adjunta su salida como dataset. "
        "Entrenar sin la partición congelada invalida la comparación entre variantes.")

keep_idx = np.load(keep_path)
assert keep_idx.ndim == 1 and len(keep_idx) > 0, "keep_indices_train.npy está vacío o mal formado"
assert keep_idx.max() < len(train_full), (
    f"Los índices llegan a {keep_idx.max()} pero train tiene {len(train_full)} imágenes. "
    "La Fase 0 se ejecutó sobre otro dataset: revisa DATA_DIR en ambos notebooks.")
assert len(np.unique(keep_idx)) == len(keep_idx), "Hay índices repetidos en la partición"


import hashlib
PARTICION_SHA = hashlib.sha256(np.ascontiguousarray(keep_idx).tobytes()).hexdigest()[:16]
print(f"Huella de la partición: {PARTICION_SHA}  ← debe coincidir en las 6 cuentas")

train_ds = Subset(train_full, keep_idx)
train_targets = np.array(train_full.targets)[keep_idx]
counts = np.bincount(train_targets, minlength=num_classes)

print(f"Train congelado: {len(train_ds)} de {len(train_full)} "
      f"({len(train_ds)/len(train_full):.1%}) | Val: {len(val_ds)} | Test: {len(test_ds)}")
print(f"Clases: {num_classes} | ratio mayor/menor: {counts.max()/max(1, counts.min()):.1f}x "
      f"| clase más pequeña: {counts.min()}")

sampler = None
if CFG.BALANCED_SAMPLING:
    pesos = (1.0 / np.power(counts, CFG.SAMPLER_POWER))[train_targets]
    sampler = WeightedRandomSampler(torch.as_tensor(pesos, dtype=torch.double),
                                    num_samples=len(train_ds), replacement=True)

loader_kw = dict(num_workers=CFG.NUM_WORKERS, pin_memory=True,
                 persistent_workers=CFG.NUM_WORKERS > 0)
train_loader = DataLoader(train_ds, batch_size=CFG.BATCH_SIZE, sampler=sampler,
                          shuffle=sampler is None, drop_last=True,
                          generator=g_loader, worker_init_fn=seed_worker, **loader_kw)
val_loader   = DataLoader(val_ds,  batch_size=CFG.BATCH_SIZE * 2, shuffle=False, **loader_kw)
test_loader  = DataLoader(test_ds, batch_size=CFG.BATCH_SIZE * 2, shuffle=False, **loader_kw)

## 5. Modelo, optimizador y planificador

In [ ]:
model = Clasificador(VARIANTE["timm"], num_classes, usa_cbam=VARIANTE["cbam"],
                     pretrained=True, drop_path=CFG.DROP_PATH, dropout=CFG.DROPOUT)
model = model.to(device).to(memory_format=torch.channels_last)

N_PARAMS = sum(p.numel() for p in model.parameters())
print(f"{CFG.MODEL_VARIANT}: {VARIANTE['timm']} | CBAM: {VARIANTE['cbam']}")
print(f"num_features: {model.num_features} | parámetros: {N_PARAMS/1e6:.2f}M")


def param_groups(model):
    g = {k: [] for k in ("bb_d", "bb_n", "hd_d", "hd_n")}
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        parte = "bb" if name.startswith("backbone.") else "hd"
        tipo  = "n" if (p.ndim <= 1 or name.endswith(".bias")) else "d"
        g[f"{parte}_{tipo}"].append(p)
    return [
        {"params": g["bb_d"], "lr": CFG.LR_BACKBONE, "weight_decay": CFG.WEIGHT_DECAY},
        {"params": g["bb_n"], "lr": CFG.LR_BACKBONE, "weight_decay": 0.0},
        {"params": g["hd_d"], "lr": CFG.LR_HEAD,     "weight_decay": CFG.WEIGHT_DECAY},
        {"params": g["hd_n"], "lr": CFG.LR_HEAD,     "weight_decay": 0.0},
    ]

optimizer = torch.optim.AdamW(param_groups(model), betas=(0.9, 0.999))

steps_per_epoch = len(train_loader)
total_steps  = CFG.EPOCHS * steps_per_epoch
warmup_steps = CFG.WARMUP_EPOCHS * steps_per_epoch

def lr_lambda(step):
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    p = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return CFG.MIN_LR_RATIO + (1 - CFG.MIN_LR_RATIO) * 0.5 * (1 + math.cos(math.pi * p))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)   # por batch, no por época

mixup_fn = Mixup(mixup_alpha=CFG.MIXUP_ALPHA, cutmix_alpha=CFG.CUTMIX_ALPHA,
                 prob=CFG.MIXUP_PROB, switch_prob=0.5, mode="batch",
                 label_smoothing=CFG.LABEL_SMOOTHING, num_classes=num_classes)
criterion_train = SoftTargetCrossEntropy()
criterion_eval  = nn.CrossEntropyLoss()

scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
ema    = ModelEmaV3(model, decay=CFG.EMA_DECAY, use_warmup=True)

## 6. Bucle de entrenamiento

In [ ]:
def train_one_epoch(epoch):
    model.train()
    mixup_fn.mixup_enabled = epoch < CFG.EPOCHS - CFG.MIXUP_OFF_EPOCHS
    suma, aciertos, vistos = 0.0, 0, 0

    for x, y in train_loader:
        x = x.to(device, non_blocking=True).to(memory_format=torch.channels_last)
        y = y.to(device, non_blocking=True)
        x_mix, y_soft = mixup_fn(x, y)

        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            out  = model(x_mix)
            loss = criterion_train(out, y_soft)

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), CFG.GRAD_CLIP)
        scaler.step(optimizer); scaler.update()
        scheduler.step()
        ema.update(model)

        suma += loss.item() * x.size(0); vistos += x.size(0)
        if not mixup_fn.mixup_enabled:
            aciertos += (out.argmax(1) == y).sum().item()

    # Con MixUp activo la accuracy de entrenamiento NO es comparable con la de
    # validacion, porque se calcula sobre imagenes mezcladas (responde a R5-m1).
    acc = aciertos / vistos if not mixup_fn.mixup_enabled else None
    return suma / vistos, acc


@torch.no_grad()
def predict_logits(net, loader):
    net.eval()
    logits, labels = [], []
    for x, y in loader:
        x = x.to(device, non_blocking=True).to(memory_format=torch.channels_last)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            logits.append(net(x).float().cpu())
        labels.append(y)
    return torch.cat(logits), torch.cat(labels)


def metricas(logits, labels):
    preds = logits.argmax(1)
    return {"loss": criterion_eval(logits, labels).item(),
            "acc": (preds == labels).float().mean().item(),
            "macro_f1": f1_score(labels.numpy(), preds.numpy(),
                                 average="macro", zero_division=0)}

In [ ]:
BEST_PATH = os.path.join(CFG.OUT_DIR, "best_model.pth")
LAST_PATH = os.path.join(CFG.OUT_DIR, "last_checkpoint.pth")

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [],
           "val_macro_f1": [], "lr": [], "segundos": []}
start_epoch, best_f1, bad, best_state, best_epoch = 0, -1.0, 0, None, -1

if CFG.RESUME_PATH and os.path.exists(CFG.RESUME_PATH):
    ck = torch.load(CFG.RESUME_PATH, map_location=device, weights_only=False)
    assert ck["variant"] == CFG.MODEL_VARIANT, "El checkpoint es de otra variante"
    model.load_state_dict(ck["model"]); ema.module.load_state_dict(ck["ema"])
    optimizer.load_state_dict(ck["optimizer"]); scheduler.load_state_dict(ck["scheduler"])
    scaler.load_state_dict(ck["scaler"])
    start_epoch, best_f1, bad = ck["epoch"] + 1, ck["best_f1"], ck["bad"]
    history, best_state, best_epoch = ck["history"], ck["best_state"], ck["best_epoch"]
    if best_state is not None:
        torch.save(best_state, BEST_PATH)    
    print(f"Reanudando desde la época {start_epoch + 1}")

t_inicio = time.time()
for epoch in range(start_epoch, CFG.EPOCHS):
    t0 = time.time()
    tr_loss, tr_acc = train_one_epoch(epoch)
    vm = metricas(*predict_logits(ema.module, val_loader))      # se valida el modelo EMA

    history["train_loss"].append(tr_loss); history["train_acc"].append(tr_acc)
    history["val_loss"].append(vm["loss"]); history["val_acc"].append(vm["acc"])
    history["val_macro_f1"].append(vm["macro_f1"])
    history["lr"].append(optimizer.param_groups[0]["lr"])
    history["segundos"].append(round(time.time() - t0, 1))

    acc_txt = f"{tr_acc:.4f}" if tr_acc is not None else "  —   "
    print(f"Ep {epoch+1:02d}/{CFG.EPOCHS} | mixup {'on ' if mixup_fn.mixup_enabled else 'off'} "
          f"| train loss {tr_loss:.4f} acc {acc_txt} | val loss {vm['loss']:.4f} "
          f"acc {vm['acc']:.4f} F1 {vm['macro_f1']:.4f} "
          f"| lr {optimizer.param_groups[0]['lr']:.2e} | {time.time()-t0:.0f}s")

    if vm["macro_f1"] > best_f1 + 1e-4:
        best_f1, bad, best_epoch = vm["macro_f1"], 0, epoch + 1
        best_state = {k: v.detach().cpu().clone() for k, v in ema.module.state_dict().items()}
        torch.save(best_state, BEST_PATH)
        print(f"   ↳ mejor modelo (val macro-F1 {best_f1:.4f})")
    else:
        bad += 1

    torch.save({"epoch": epoch, "best_f1": best_f1, "bad": bad, "history": history,
                "best_state": best_state, "best_epoch": best_epoch,
                "model": model.state_dict(), "ema": ema.module.state_dict(),
                "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
                "scaler": scaler.state_dict(), "variant": CFG.MODEL_VARIANT}, LAST_PATH)

    if bad >= CFG.PATIENCE:
        print(f"Early stopping en la época {epoch+1}")
        break

MINUTOS = (time.time() - t_inicio) / 60
print(f"\nMejor val macro-F1: {best_f1:.4f} (época {best_epoch}) | {MINUTOS:.0f} min")

## 7. Logits para McNemar

In [ ]:
best_model = Clasificador(VARIANTE["timm"], num_classes, usa_cbam=VARIANTE["cbam"],
                          pretrained=False, dropout=CFG.DROPOUT)
best_model.load_state_dict(torch.load(BEST_PATH, map_location="cpu"))
best_model = best_model.to(device).to(memory_format=torch.channels_last).eval()

for nombre, loader in (("test", test_loader), ("val", val_loader)):
    lg, lb = predict_logits(best_model, loader)
    np.savez_compressed(os.path.join(CFG.OUT_DIR, f"logits_{nombre}.npz"),
                        logits=lg.numpy().astype(np.float32),
                        labels=lb.numpy().astype(np.int16),
                        variant=CFG.MODEL_VARIANT)
    m = metricas(lg, lb)
    print(f"{nombre:5s} | acc {m['acc']:.4f} | macro-F1 {m['macro_f1']:.4f}")
    if nombre == "test":
        test_metrics = m

### Evaluación del entrenamiento sin aumento ni MixUp


In [ ]:
TRAIN_LIMPIO = None
if CFG.EVAL_TRAIN_LIMPIO:
    train_eval = Subset(datasets.ImageFolder(os.path.join(CFG.DATA_DIR, "train"),
                                             transform=val_tf), keep_idx)
    lg, lb = predict_logits(best_model, DataLoader(train_eval,
                                                   batch_size=CFG.BATCH_SIZE * 2,
                                                   shuffle=False, **loader_kw))
    TRAIN_LIMPIO = metricas(lg, lb)
    ultima_mixup = next((a for a in reversed(history["train_acc"]) if a is not None), None)
    print(f"Train sin aumento ni MixUp : acc {TRAIN_LIMPIO['acc']:.4f} "
          f"| macro-F1 {TRAIN_LIMPIO['macro_f1']:.4f}")
    if ultima_mixup is not None:
        print(f"Train con MixUp activo     : acc {ultima_mixup:.4f}  (NO comparable)")
    print(f"Val (referencia)           : acc {history['val_acc'][-1]:.4f}")

## 8. Latencia de inferencia

In [ ]:
def medir_latencia(bs, n=50, calentar=15):
    x = torch.randn(bs, 3, CFG.IMG_SIZE, CFG.IMG_SIZE,
                    device=device).to(memory_format=torch.channels_last)
    with torch.no_grad():
        for _ in range(calentar):
            best_model(x)
        if device.type == "cuda":
            torch.cuda.synchronize()
        t = []
        for _ in range(n):
            t0 = time.perf_counter()
            best_model(x)
            if device.type == "cuda":
                torch.cuda.synchronize()
            t.append((time.perf_counter() - t0) * 1000)
    t = np.array(t)
    return {"media_ms": float(t.mean()), "p50_ms": float(np.percentile(t, 50)),
            "p95_ms": float(np.percentile(t, 95)),
            "por_imagen_ms": float(t.mean() / bs),
            "imagenes_por_s": float(bs * 1000 / t.mean())}

LAT = medir_latencia(1)
LAT["tamano_mb"] = os.path.getsize(BEST_PATH) / 1e6
LAT["por_lote"] = {str(bs): medir_latencia(bs) for bs in (1, 8, 32)}

try:
    from torch.utils.flop_counter import FlopCounterMode
    fc = FlopCounterMode(display=False)
    with fc, torch.no_grad():
        best_model(torch.randn(1, 3, CFG.IMG_SIZE, CFG.IMG_SIZE,
                               device=device).to(memory_format=torch.channels_last))
    LAT["gflops"] = fc.get_total_flops() / 1e9
except Exception as e:
    LAT["gflops"] = None
    print(f"No se pudieron contar los FLOPs ({type(e).__name__})")

MEM_PICO_GB = (torch.cuda.max_memory_allocated() / 1e9 if device.type == "cuda" else 0.0)

print(f"Latencia batch=1 — media {LAT['media_ms']:.1f} ms | p95 {LAT['p95_ms']:.1f} ms")
print(f"Rendimiento batch=32 — {LAT['por_lote']['32']['imagenes_por_s']:.0f} img/s "
      f"({LAT['por_lote']['32']['por_imagen_ms']:.2f} ms por imagen)")
print(f"GFLOPs: {LAT['gflops']:.2f}" if LAT["gflops"] else "GFLOPs: no disponible")
print(f"Modelo {LAT['tamano_mb']:.0f} MB | GPU {gpu_name} | memoria pico {MEM_PICO_GB:.2f} GB")

## 9. Configuración completa y empaquetado

In [ ]:
config = {
    "variante": CFG.MODEL_VARIANT,
    "backbone_timm": VARIANTE["timm"],
    "usa_cbam": VARIANTE["cbam"],
    "num_features": int(model.num_features),
    "parametros": int(N_PARAMS),

    "datos": {
        "dataset": "Niayes Crops Disease V2",
        "train_original": len(train_full), "train_congelado": len(train_ds),
        "val": len(val_ds), "test": len(test_ds),
        "num_classes": num_classes, "class_names": class_names,
        "imagenes_por_clase_train": counts.tolist(),
        "particion_congelada": "keep_indices_train.npy (Fase 0)",
        "particion_sha256_16": PARTICION_SHA,
    },

    "entrada": {"resize": CFG.RESIZE, "center_crop": CFG.IMG_SIZE,
                "mean": list(CFG.MEAN), "std": list(CFG.STD)},

    "aumento_datos": {
        "random_zoom_out": {"p": 0.2, "side_range": [1.0, 1.8]},
        "random_resized_crop": {"scale": [0.3, 1.0], "size": CFG.IMG_SIZE},
        "horizontal_flip_p": 0.5, "vertical_flip_p": 0.2,
        "rotation": {"p": 0.5, "grados": 30},
        "perspective": {"p": 0.2, "distortion_scale": 0.2},
        "color_jitter": {"p": 0.8, "brightness": 0.4, "contrast": 0.4,
                         "saturation": 0.3, "hue": 0.05},
        "grayscale_p": 0.05,
        "gaussian_blur": {"p": 0.3, "kernel": 9, "sigma": [0.1, 3.0]},
        "jpeg": ({"p": 0.3, "quality": [30, 95]} if TIENE_JPEG else None),
        "random_erasing": {"p": 0.25, "scale": [0.02, 0.2], "ratio": [0.3, 3.3], "value": 0},
    },

    "mezcla_lotes": {"mixup_alpha": CFG.MIXUP_ALPHA, "cutmix_alpha": CFG.CUTMIX_ALPHA,
                     "prob": CFG.MIXUP_PROB, "switch_prob": 0.5, "mode": "batch",
                     "desactivado_ultimas_epocas": CFG.MIXUP_OFF_EPOCHS},

    "semillas": {"global": CFG.SEED, "generador_dataloader": CFG.SEED,
                 "worker_init_fn": "derivada de torch.initial_seed()"},

    "desbalance": {"estrategia": "WeightedRandomSampler",
                   "peso": f"1 / n^{CFG.SAMPLER_POWER}",
                   "replacement": True, "num_samples": len(train_ds),
                   "perdida_ponderada_por_clase": False},

    "optimizacion": {
        "optimizador": "AdamW", "betas": [0.9, 0.999],
        "lr_backbone": CFG.LR_BACKBONE, "lr_head": CFG.LR_HEAD,
        "weight_decay": CFG.WEIGHT_DECAY,
        "sin_decay_en": "normalizacion y sesgos",
        "scheduler": "warmup lineal + coseno, actualizado por batch",
        "warmup_epochs": CFG.WARMUP_EPOCHS, "min_lr_ratio": CFG.MIN_LR_RATIO,
        "label_smoothing": CFG.LABEL_SMOOTHING, "grad_clip": CFG.GRAD_CLIP,
        "batch_size": CFG.BATCH_SIZE, "epochs_max": CFG.EPOCHS,
        "early_stopping": {"metrica": "val macro-F1", "patience": CFG.PATIENCE},
        "ema_decay": CFG.EMA_DECAY, "amp": USE_AMP,
        "dropout": CFG.DROPOUT, "drop_path": CFG.DROP_PATH,
        "busqueda_de_hiperparametros": "ninguna; misma receta para las 6 arquitecturas",
    },

    "resultado": {
        "mejor_epoca": best_epoch, "epocas_ejecutadas": len(history["train_loss"]),
        "val_macro_f1": best_f1,
        "test_accuracy": test_metrics["acc"], "test_macro_f1": test_metrics["macro_f1"],
        "train_limpio_accuracy": (TRAIN_LIMPIO["acc"] if TRAIN_LIMPIO else None),
        "train_limpio_macro_f1": (TRAIN_LIMPIO["macro_f1"] if TRAIN_LIMPIO else None),
        "train_acc_con_mixup_no_comparable": next(
            (a for a in reversed(history["train_acc"]) if a is not None), None),
        "minutos_entrenamiento": round(MINUTOS, 1),
        "segundos_por_epoca": history["segundos"],
        "latencia": LAT,
    },

    "entorno": {
        "gpu": gpu_name, "gpu_memoria_total_gb": round(gpu_mem, 1),
        "gpu_memoria_pico_gb": round(MEM_PICO_GB, 2),
        "python": platform.python_version(), "torch": torch.__version__,
        "timm": timm.__version__, "cuda": torch.version.cuda,
        "semilla": CFG.SEED,
    },
}

with open(os.path.join(CFG.OUT_DIR, "config.json"), "w") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)
with open(os.path.join(CFG.OUT_DIR, "history.json"), "w") as f:
    json.dump(history, f, indent=2)

os.remove(LAST_PATH)   

salidas = ["best_model.pth", "logits_test.npz", "logits_val.npz",
           "config.json", "history.json"]
zip_name = f"resultados_{CFG.MODEL_VARIANT}.zip"
with zipfile.ZipFile(os.path.join(CFG.OUT_DIR, zip_name), "w", zipfile.ZIP_DEFLATED) as z:
    for f_ in salidas:
        p = os.path.join(CFG.OUT_DIR, f_)
        if os.path.exists(p):
            z.write(p, f_)

print(json.dumps({k: v for k, v in config["resultado"].items()
                  if k not in ("segundos_por_epoca", "latencia")},
                 indent=2, ensure_ascii=False))
print("\n--- pegar esto en el hilo del equipo ---")
print(f"{CFG.MODEL_VARIANT}: acc {test_metrics['acc']:.4f} | "
      f"macro-F1 {test_metrics['macro_f1']:.4f} | {N_PARAMS/1e6:.1f}M par. | "
      f"{LAT['media_ms']:.1f} ms | partición {PARTICION_SHA}")
print(f"\n{zip_name}")
print("Output → New Dataset, nómbralo:  plantnova-" + CFG.MODEL_VARIANT.replace("_", "-"))

In [ ]:
ep = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
axes[0].plot(ep, history["train_loss"], label="Entrenamiento")
axes[0].plot(ep, history["val_loss"], label="Validación")
axes[0].set_title("Pérdida"); axes[0].legend()
axes[1].plot(ep, history["val_acc"], label="Accuracy (val)")
axes[1].plot(ep, history["val_macro_f1"], label="Macro-F1 (val)")
tr = [a if a is not None else np.nan for a in history["train_acc"]]
if np.isfinite(tr).any():
    axes[1].plot(ep, tr, "o--", ms=4, label="Accuracy (train, sin mezcla)")
axes[1].axvline(best_epoch, ls="--", c="gray", lw=1)
axes[1].set_title("Validación"); axes[1].legend()
axes[2].plot(ep, history["lr"]); axes[2].set_title("Tasa de aprendizaje")
for ax in axes:
    ax.set_xlabel("Época"); ax.grid(alpha=0.3)
fig.suptitle(f"{CFG.MODEL_VARIANT} — mejor época {best_epoch}", fontweight="bold")
fig.savefig(os.path.join(CFG.OUT_DIR, f"curvas_{CFG.MODEL_VARIANT}.png"), dpi=200,
            bbox_inches="tight")
plt.show()